# ML baselines — DL과 동일한 nested 5-fold CV 로 재평가

**목적**: Random Forest / SVM / XGBoost 를 genotype-only CNN·multimodal 과 **같은 분할**로 평가해
Table 1에 pooled / mean±SD 를 넣고, fold별 paired 비교가 가능하게 한다.

**DL과 맞춘 것 (동일 프로토콜)**
- 같은 데이터: 전체 `dataset.csv` 를 한 번에 로드
- 같은 분할: `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` → **각 fold가 정확히 한 번씩 최종 test**
- 각 fold에서 나머지 4 fold로 학습 → held-out test fold 평가 → **모든 샘플이 정확히 한 번 평가(out-of-fold)**
- fold별 지표를 모아 **mean ± SD (ddof=1)** + pooled out-of-fold
- averaging = **macro** — DL과 동일

**모델 특성상 다른 것 (불가피 — Methods에 명시)**
- 입력: ML은 (두 allele one-hot 합산 → flatten → **fold-train에만 fit한 TruncatedSVD**) ∥ **ECFP(1024)**.
- **early stopping 없음**: 고전 ML 은 validation 기반 early stopping/checkpoint 개념이 없어,
  DL의 "4 fold 내부 train/val 분할"에 해당하는 과정 없이 **4 fold 전체로 학습**한다.
- **누수 방지**: SVD / StandardScaler 는 매 fold의 **trainval(4 fold)에만 fit**, test fold에는 transform만.

출력: `ml_per_fold_test.csv` (model, fold, Accuracy, Precision, Recall, F1, AUROC, AUPR).

In [18]:
import os, numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs


In [26]:
# ===== Config =====
ONEHOT_FOLDER = 'onehot_7'
DATA_CSV = 'dataset.csv'          # 전체 데이터 (nested 5-fold CV)
TARGET_ROWS_SINGLE = 4330
TARGET_COLS = 7
ECFP_BITS = 1024
SVD_MAX_COMPONENTS = 500
RANDOM_STATE = 42
N_CLASSES = 3
N_FOLDS = 5                         # 5-fold (DL과 동일)
METRICS = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUROC', 'AUPR']
AVERAGE = 'macro'                  # DL과 동일

In [27]:
# ===== 전체 데이터 로드 + 라벨 인코딩 =====
all_data = pd.read_csv(DATA_CSV)
label_encoder = LabelEncoder()
all_data['Class'] = label_encoder.fit_transform(all_data['Class'])
print('label mapping:', {i: c for i, c in enumerate(label_encoder.classes_)})
print('전체 샘플 수:', len(all_data))
print(all_data['Class'].value_counts().sort_index())

label mapping: {0: 'de', 1: 'f', 2: 'nor'}
전체 샘플 수: 622
0    250
1    195
2    177
Name: Class, dtype: int64


In [28]:
# ===== one-hot 로드 + 두 allele 합산 =====
import torch
onehot_matrices = {}
for file in os.listdir(ONEHOT_FOLDER):
    if not file.endswith('.csv'):
        continue
    var_key = file.split('.csv')[0].replace('_', '.')
    try:
        mat = pd.read_csv(os.path.join(ONEHOT_FOLDER, file), header=None).values
        padded = np.zeros((TARGET_ROWS_SINGLE, TARGET_COLS), dtype=np.float32)
        r = min(mat.shape[0], TARGET_ROWS_SINGLE); c = min(mat.shape[1], TARGET_COLS)
        padded[:r, :c] = mat[:r, :c]
        onehot_matrices[var_key] = padded
    except Exception as e:
        print(f'[WARN] {file}: {e}')

def variant_to_sum(variant_str):
    if pd.isna(variant_str):
        variants = []
    else:
        variants = [v.strip() for v in str(variant_str).split(';') if v.strip()]
    mats = []
    for v in variants[:2]:
        mats.append(onehot_matrices.get(v, np.zeros((TARGET_ROWS_SINGLE, TARGET_COLS), dtype=np.float32)))
    while len(mats) < 2:
        mats.append(np.zeros((TARGET_ROWS_SINGLE, TARGET_COLS), dtype=np.float32))
    return mats[0] + mats[1]

def seq_matrix(df):
    X = np.stack([variant_to_sum(v) for v in df['Variant']]).astype(np.float64)
    return X.reshape(len(df), -1)

X_seq_full = seq_matrix(all_data)      # (610, 4330*7)
y_full     = all_data['Class'].values
print('seq flat:', X_seq_full.shape)

seq flat: (622, 30310)


In [29]:
# ===== SMILES -> ECFP =====
def smiles_to_ecfp(smiles, n_bits=ECFP_BITS, radius=2):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros((n_bits,), dtype=np.float32)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

X_drug_full = np.vstack([smiles_to_ecfp(s) for s in all_data['isomeric']])
print('ECFP:', X_drug_full.shape)

ECFP: (622, 1024)


## 모델 정의 (best HP 고정)

In [30]:
# ===== 모델별 best 하이퍼파라미터 (기존 노트북 탐색 결과로 고정) =====
# 주: 논문 Methods/Supplementary 에 그대로 기재
def make_model(name):
    if name == 'Random forest':
        return RandomForestClassifier(
            n_estimators=250, max_depth=30, min_samples_leaf=2,
            min_samples_split=2, n_jobs=-1, random_state=RANDOM_STATE)
    if name == 'SVM':
        # StandardScaler + poly SVM (tuned). Scaler는 Pipeline 안에서 fold-train에만 fit됨
        return Pipeline([
            ('scaler', StandardScaler()),
            ('svm', SVC(kernel='poly', C=0.22788439100624497, coef0=7.261181986115988,
                        degree=4, gamma=2.6713901787135634e-05, probability=True, random_state=RANDOM_STATE))])
    if name == 'XGBoost':
        return XGBClassifier(
            n_estimators=658, learning_rate=0.048998630084050664, max_depth=5,
            subsample=0.9464704583099741, colsample_bytree=0.6624074561769746,
            objective='multi:softprob', num_class=N_CLASSES, eval_metric='mlogloss',
            tree_method='hist', random_state=RANDOM_STATE)
    raise ValueError(name)

MODEL_NAMES = ['Random forest', 'SVM', 'XGBoost']


## 10-fold 동일 프로토콜 평가

In [31]:
# =====================================================================
# Nested 5-fold Cross-Validation (ML baselines)
#   - 바깥 5-fold: 각 fold를 정확히 한 번씩 "최종 test"로 사용
#   - 나머지 4 fold 전체로 학습 (ML은 early stopping 없음)
#   - SVD / StandardScaler 는 trainval(4 fold)에만 fit → test fold transform (누수 방지)
#   - 모든 샘플이 정확히 한 번씩 test 평가 (out-of-fold)
# =====================================================================
CLASSES = [0, 1, 2]

def metrics_from_pred(y_true, y_pred, y_proba):
    y_bin = label_binarize(y_true, classes=CLASSES)
    return {
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average=AVERAGE, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, average=AVERAGE, zero_division=0),
        'F1':        f1_score(y_true, y_pred, average=AVERAGE, zero_division=0),
        'AUROC':     roc_auc_score(y_true, y_proba, multi_class='ovr', average=AVERAGE, labels=CLASSES),
        'AUPR':      average_precision_score(y_bin, y_proba, average=AVERAGE),
    }

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
n_total = len(y_full)

# 모델별 out-of-fold 예측 저장
oof = {name: {'probs': np.zeros((n_total, N_CLASSES)), 'pred': np.full(n_total, -1, int)}
       for name in MODEL_NAMES}
oof_true = y_full.copy()

rows = []
for fold, (trainval_idx, test_idx) in enumerate(skf.split(X_seq_full, y_full)):
    # --- 누수 방지: SVD 는 trainval(4 fold)에만 fit ---
    n_comp = max(2, min(SVD_MAX_COMPONENTS, len(trainval_idx) - 1, X_seq_full.shape[1] - 1))
    svd = TruncatedSVD(n_components=n_comp, random_state=RANDOM_STATE)
    svd.fit(X_seq_full[trainval_idx])

    def build(seq, drug):
        return np.hstack([svd.transform(seq), drug])

    X_tr = build(X_seq_full[trainval_idx], X_drug_full[trainval_idx])
    X_te = build(X_seq_full[test_idx],     X_drug_full[test_idx])   # held-out test fold
    y_tr = y_full[trainval_idx]
    y_te = y_full[test_idx]

    for name in MODEL_NAMES:
        clf = make_model(name)
        clf.fit(X_tr, y_tr)
        y_pred  = clf.predict(X_te)
        y_proba = clf.predict_proba(X_te)
        m = metrics_from_pred(y_te, y_pred, y_proba)
        rows.append({'model': name, 'fold': fold + 1, **m})

        oof[name]['probs'][test_idx] = y_proba
        oof[name]['pred'][test_idx]  = y_pred

    print(f'fold {fold+1}/{N_FOLDS} done (SVD comp={n_comp}, '
          f'trainval={len(trainval_idx)}, test={len(test_idx)})')

ml_df = pd.DataFrame(rows)
ml_df.to_csv('ml_per_fold_test.csv', index=False, encoding='utf-8-sig')
print('\nsaved ml_per_fold_test.csv', ml_df.shape)

fold 1/5 done (SVD comp=496, trainval=497, test=125)
fold 2/5 done (SVD comp=496, trainval=497, test=125)
fold 3/5 done (SVD comp=497, trainval=498, test=124)
fold 4/5 done (SVD comp=497, trainval=498, test=124)
fold 5/5 done (SVD comp=497, trainval=498, test=124)

saved ml_per_fold_test.csv (15, 8)


## 요약

In [32]:
# ===== 요약: 모델별 Pooled out-of-fold + fold별 mean ± SD (ddof=1) =====
def metrics_from_oof(y_true, pred, proba):
    y_bin = label_binarize(y_true, classes=CLASSES)
    return {
        'Accuracy':  accuracy_score(y_true, pred),
        'Precision': precision_score(y_true, pred, average=AVERAGE, zero_division=0),
        'Recall':    recall_score(y_true, pred, average=AVERAGE, zero_division=0),
        'F1':        f1_score(y_true, pred, average=AVERAGE, zero_division=0),
        'AUROC':     roc_auc_score(y_true, proba, multi_class='ovr', average=AVERAGE, labels=CLASSES),
        'AUPR':      average_precision_score(y_bin, proba, average=AVERAGE),
    }

summary_rows = []
for name in MODEL_NAMES:
    sub = ml_df[ml_df['model'] == name]
    pooled = metrics_from_oof(oof_true, oof[name]['pred'], oof[name]['probs'])
    print(f'\n=== {name} ===')
    print('  [Pooled out-of-fold]')
    for m in METRICS:
        print(f'    {m}: {pooled[m]:.4f}')
    print('  [Fold별 mean ± SD (ddof=1)]')
    row = {'model': name}
    for m in METRICS:
        mean, sd = sub[m].mean(), sub[m].std(ddof=1)
        print(f'    {m}: {mean:.4f} ± {sd:.4f}')
        row[f'{m}_mean'] = mean; row[f'{m}_sd'] = sd; row[f'{m}_pooled'] = pooled[m]
    summary_rows.append(row)

pd.DataFrame(summary_rows).to_csv('ml_summary_mean_sd.csv', index=False, encoding='utf-8-sig')
print('\nsaved ml_summary_mean_sd.csv')


=== Random forest ===
  [Pooled out-of-fold]
    Accuracy: 0.8312
    Precision: 0.8464
    Recall: 0.8257
    F1: 0.8334
    AUROC: 0.9540
    AUPR: 0.9189
  [Fold별 mean ± SD (ddof=1)]
    Accuracy: 0.8312 ± 0.0119
    Precision: 0.8475 ± 0.0096
    Recall: 0.8257 ± 0.0152
    F1: 0.8330 ± 0.0122
    AUROC: 0.9549 ± 0.0095
    AUPR: 0.9225 ± 0.0174

=== SVM ===
  [Pooled out-of-fold]
    Accuracy: 0.7283
    Precision: 0.7405
    Recall: 0.7475
    F1: 0.7313
    AUROC: 0.8917
    AUPR: 0.7834
  [Fold별 mean ± SD (ddof=1)]
    Accuracy: 0.7282 ± 0.0777
    Precision: 0.7445 ± 0.0719
    Recall: 0.7474 ± 0.0721
    F1: 0.7295 ± 0.0791
    AUROC: 0.8964 ± 0.0420
    AUPR: 0.8050 ± 0.0678

=== XGBoost ===
  [Pooled out-of-fold]
    Accuracy: 0.7990
    Precision: 0.8011
    Recall: 0.8166
    F1: 0.8016
    AUROC: 0.9364
    AUPR: 0.8939
  [Fold별 mean ± SD (ddof=1)]
    Accuracy: 0.7991 ± 0.0778
    Precision: 0.8087 ± 0.0711
    Recall: 0.8166 ± 0.0670
    F1: 0.7997 ± 0.0808
    AUROC:

---
**다음**
- `ml_per_fold_test.csv` 를 `cnn_per_fold_test.csv`, `multimodal_per_fold_test.csv` 와 같은 폴더에 두고
  paired test 를 돌리면 Multimodal vs 각 모델 fold별 paired p-value 를 얻을 수 있다.
  (세 노트북 모두 `dataset.csv` 에 `StratifiedKFold(5, shuffle=True, random_state=42)` 를 적용하므로
  fold k 의 test 샘플 구성이 동일 → paired 비교가 유효하다.)
- 입력 표현 차이(ML: SVD∥ECFP / DL: one-hot·graph)와 '동일 분할(같은 5-fold, macro, SD, paired test)',
  그리고 'ML은 early stopping 미적용'을 Methods에 한 문단으로 기재.